## Preprocessing of data for perturbgen
### data formatting / curation

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import matplotlib.pyplot as plt
import os
import anndata
import gc
from scipy.sparse import csr_matrix

seed = 42
np.random.seed(seed)


main_path = "/rds/general/user/ap5625/home/perturbation_modeling"

extra_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/T2D_REM/Phase_2/objects"

input_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/Ada/perturbation_modeling"

In [2]:
adata = sc.read_h5ad(input_path + "/adipocytes_subset_2k.h5ad")

In [3]:
print(adata)

AnnData object with n_obs × n_vars = 2000 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0.5

In [4]:
adata.var_names_make_unique()
adata.obs_names_make_unique()

In [5]:
adata.X = adata.layers["raw"].copy()

In [6]:
adata.X.max()

463.0

In [7]:
print(adata)

AnnData object with n_obs × n_vars = 2000 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0.5

In [9]:
# check genes in geneformer
import pickle

# Load the Geneformer gene→token mapping - they have it in their code
with open(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl",
    "rb",
) as f:
    gf_genes = pickle.load(f)

In [10]:
# Keep only genes present in the Geneformer vocabulary
before = adata.n_vars

In [11]:
adata.var["in_gf"] = adata.var_names.isin(
    gf_genes.values()
) | adata.var_names.isin(gf_genes.keys())
adata = adata[:, adata.var["in_gf"]].copy()

In [12]:
print(f"Genes: {before} → {adata.n_vars} after Geneformer filtering")

Genes: 2000 → 1979 after Geneformer filtering


In [14]:
# also genes with very low expression (near zero) - do not get added in the tokenisation dictionary

# On your Geneformer-filtered adata (raw counts in X or layers['raw'])
counts = adata.layers["raw"] if "raw" in adata.layers else adata.X
genes_expressed = np.asarray((counts > 0).sum(axis=0)).flatten() > 0
print(
    f"Genes with non-zero expression in ≥1 cell: {genes_expressed.sum()} / {adata.n_vars}"
)

# Genes expressed in a meaningful number of cells
genes_10cells = np.asarray((counts > 0).sum(axis=0)).flatten() >= 10
print(f"Genes expressed in ≥10 cells: {genes_10cells.sum()} / {adata.n_vars}")

Genes with non-zero expression in ≥1 cell: 1558 / 1979
Genes expressed in ≥10 cells: 1160 / 1979


# Ensemble ID mapping

In [10]:
# map to ENSEMBL IDs

ref = pd.read_csv(main_path + "/mart_export.txt", sep="\t")

print(ref.columns.tolist())

['Gene stable ID', 'Gene name']


In [11]:
symbol_to_ensembl = dict(zip(ref["Gene name"], ref["Gene stable ID"]))


def map_to_ensembl(gene_name):
    if str(gene_name).startswith("ENSG"):
        return gene_name
    else:
        return symbol_to_ensembl.get(gene_name, None)


adata.var["ensembl_id"] = [map_to_ensembl(g) for g in adata.var_names]

missing = adata.var["ensembl_id"].isna().sum()
print(f"Still unmatched: {missing}")
print(adata.var[adata.var["ensembl_id"].isna()].index.tolist()[:20])

adata = adata[:, adata.var["ensembl_id"].notna()].copy()
adata.var_names = adata.var["ensembl_id"].values
print(f"Final gene count: {adata.n_vars}")

Still unmatched: 26
['C1orf109', 'LEXM', 'C17orf97', 'CCDC103', 'LINC00298', 'PDK1-AS1', 'ELFN2-1', 'CSNKA2IP', 'ALG14-AS1', 'RIPK2-DT', 'LINC00443', 'KIAA2026', 'LINC01357', 'C17orf49', 'LINC01238-1', 'OR2B8P', 'TSPEAR-AS2', 'LINC03023-1', 'CENPJ', 'C18orf25']
Final gene count: 1974


In [12]:
ensembl_to_symbol = dict(zip(ref["Gene stable ID"], ref["Gene name"]))

adata.var["gene_symbol"] = adata.var_names.map(ensembl_to_symbol)

# ensembl_id column is already set as var_names but add it as a column too
adata.var["ensembl_id"] = adata.var_names

# Verify
print(adata.var[["ensembl_id", "gene_symbol"]].head(10))

                      ensembl_id gene_symbol
ENSG00000244879  ENSG00000244879  GABPB1-AS1
ENSG00000225269  ENSG00000225269   LINC00705
ENSG00000174611  ENSG00000174611          KY
ENSG00000139352  ENSG00000139352       ASCL1
ENSG00000232455  ENSG00000232455   LARS2-AS1
ENSG00000134242  ENSG00000134242      PTPN22
ENSG00000081800  ENSG00000081800     SLC13A1
ENSG00000095564  ENSG00000095564       BTAF1
ENSG00000233492  ENSG00000233492   BETALINC1
ENSG00000153147  ENSG00000153147     SMARCA5


In [13]:
print(f"Total genes: {adata.n_vars}")
print(f"Genes with symbol: {adata.var['gene_symbol'].notna().sum()}")

Total genes: 1974
Genes with symbol: 1974


In [14]:
# 5. Set var_names to ensembl_id
adata.var_names = adata.var["ensembl_id"]

In [15]:
print(f"adata now has {adata.n_vars} genes with Ensembl IDs as var_names.")

adata now has 1974 genes with Ensembl IDs as var_names.


In [16]:
adata.var.index.name = "gene_id"

In [17]:
adata.var

,highly_variable,means,dispersions,dispersions_norm,highly_variable_nbatches,highly_variable_intersection,mean,std,ensembl_id,gene_symbol
gene_id,,,,,,,,,,
ENSG00000244879,False,2.319541e-01,1.476932,-0.253013,0,False,8.109487e-11,0.624937,ENSG00000244879,GABPB1-AS1
ENSG00000225269,False,2.501364e-03,1.650039,-0.098541,0,False,-6.286545e-13,0.043772,ENSG00000225269,LINC00705
ENSG00000174611,False,1.350673e-02,1.535456,-0.119745,0,False,-1.143743e-11,0.103891,ENSG00000174611,KY
ENSG00000139352,False,9.746840e-07,1.214197,-1.123107,0,False,7.780403e-14,0.003478,ENSG00000139352,ASCL1
ENSG00000232455,False,1.562263e-02,1.616517,0.040035,0,False,9.000068e-12,0.103082,ENSG00000232455,LARS2-AS1
...,...,...,...,...,...,...,...,...,...,...
ENSG00000244682,False,6.418056e-04,1.943121,0.220126,0,False,3.112101e-12,0.025638,ENSG00000244682,FCGR2C
ENSG00000136943,False,5.841249e-03,1.611765,-0.114225,0,False,-8.760381e-12,0.076072,ENSG00000136943,CTSV
ENSG00000144451,False,3.875462e-01,1.630630,0.282248,0,False,-7.233974e-11,0.754632,ENSG00000144451,SPAG16


## obs preprocessing

In [119]:
print(adata.obs.columns.tolist())

['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'Remission_2021', 'Age', 'Ethnicity', 'F

In [18]:
# only include obese and weight loss for the start - as they are paired


adata_sub = adata[
    adata.obs["condition"].isin(["baseline", "weightloss"])
].copy()

In [19]:
print(adata.obs["condition"].value_counts())
print(adata_sub.obs["condition"].value_counts())

condition
weightloss    1110
baseline       792
Lean            98
Name: count, dtype: int64
condition
weightloss    1110
baseline       792
Name: count, dtype: int64


In [20]:
baseline_donors = set(
    adata_sub.obs[adata_sub.obs["condition"] == "baseline"]["Donor"].unique()
)
weight_loss_donors = set(
    adata_sub.obs[adata_sub.obs["condition"] == "weightloss"]["Donor"].unique()
)

In [21]:
len(baseline_donors)

63

In [22]:
len(weight_loss_donors)

59

In [23]:
paired = baseline_donors & weight_loss_donors

In [24]:
len(paired)

56

In [25]:
# Keep only paired donors
adata_sub2 = adata_sub[adata_sub.obs["Donor"].isin(paired)].copy()

In [26]:
print(f"Cells after keeping paired donors only: {adata_sub2.n_obs}")

Cells after keeping paired donors only: 1847


In [27]:
adata_sub2.obs["donor_id"] = adata_sub2.obs["Donor"]

In [28]:
adata_sub2.obs["condition"] = adata_sub2.obs["condition"].map(
    {"baseline": "obese", "weightloss": "weightloss"}
)

In [29]:
adata_sub2.obs["cell_states_adipocytes"] = adata_sub2.obs["cell_state_t2d"]

In [30]:
print(adata_sub2.var["highly_variable"].sum())

144


In [31]:
adata_sub2.write(input_path + "/preprocessed_adipocytes_subset_2k.h5ad")

In [152]:
print(adata_sub2.obs["condition"].value_counts())

condition
weightloss    1087
obese          760
Name: count, dtype: int64


In [154]:
print(adata_sub2)

AnnData object with n_obs × n_vars = 1847 × 1974
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_0.5

In [155]:
print(adata_sub2.obs["cell_states_adipocytes"].value_counts())

cell_states_adipocytes
AD2            515
AD3            504
AD1            317
AD4            157
AD6            119
AD9             85
AD10            54
AD5_basal       33
AD5_BMPERhi     26
AD7             19
AD8             14
Unassigned       4
Name: count, dtype: int64


In [157]:
print(adata_sub2.obs["condition"].value_counts())

condition
weightloss    1087
obese          760
Name: count, dtype: int64


In [158]:
print(adata_sub2.obs["donor_id"].value_counts())

donor_id
DR031    153
DR053    111
DR052    106
DR039     98
DR058     98
DR043     97
DR034     93
DR064     88
DR062     75
DR057     75
DR020     68
DR014     55
DR042     52
MMT09     44
DR045     43
DR040     42
DR055     41
DR065     41
DR032     37
MMT12     33
P104B     28
DR011     26
DR051     24
P94B      23
MMT11     20
MMT22     18
DR015     17
DR033     16
P26B      14
P157B     14
P140B     14
P93B      13
DR016     12
P156B     12
P74B      11
MMT14     11
DR006     10
P53B      10
P80B       9
P96B       9
P78B       8
DR008      8
P95B       7
P73B       7
DR012      7
P85B       7
P20B       6
MMT02      5
P47B       5
MMT07      5
DR005      5
MMT23      4
DR001      4
P87B       3
P22B       3
MMT15      2
Name: count, dtype: int64
